# 46 — Energy proxy that refuses to overclaim

Event-driven vs clock-driven **register toggle proxy** using a small recurrent
population of catalogue **Perfect Integrator** neurons (polyglot-complete). Reports
**toggle counts**, not joules.

## Honesty box

| | |
|---|---|
| **Proves** | Toggle-count proxy scales with N and activity; event-driven can reduce toggles vs worst-case clock-driven α=1. |
| **Does not prove** | Silicon power, energy in joules, PPA, or FPGA power reports. |
| **Artefacts** | Local figures only. |
| **Models** | `PerfectIntegratorNeuron` via `Population` / `Network` (polyglot-complete; drives clean spikes under Poisson). |



In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore.network.monitor import SpikeMonitor
from sc_neurocore.network.network import Network
from sc_neurocore.network.population import Population
from sc_neurocore.network.projection import Projection
from sc_neurocore.network.stimulus import PoissonInput
from sc_neurocore.neurons.models import PerfectIntegratorNeuron

print("SC-NeuroCore — NB-46 honest energy proxy")
REGS_PER_NEURON = 5  # didactic register count
DT = 0.1  # ms


In [ ]:
def run_toggles(n: int, duration_s: float = 50.0, rate_hz: float = 60.0, seed: int = 0):
    pop = Population(PerfectIntegratorNeuron, n=n, label="pi")
    proj = Projection(pop, pop, weight=0.25, probability=0.08, seed=seed)
    drive = PoissonInput(n=n, rate_hz=rate_hz, weight=1.5, dt=DT, seed=seed + 1)
    mon = SpikeMonitor(pop, label="m")
    net = Network(pop, proj, drive, mon)
    net.run(duration=duration_s, dt=DT)
    steps = int(duration_s / (DT / 1000.0)) if DT > 1 else int(duration_s / DT)
    # SpikeMonitor times are in seconds when duration is seconds — match notebook 20 style
    # Recompute steps from duration/dt assuming dt seconds if dt<1
    steps = max(int(round(duration_s / DT)), 1)
    activity = np.zeros(steps)
    for _nid, times in mon.spike_trains.items():
        for t in times:
            si = int(t / DT)
            if 0 <= si < steps:
                activity[si] += 1
    clock = n * REGS_PER_NEURON * steps
    event = int(np.sum(activity)) * REGS_PER_NEURON
    mean_rate = mon.count / (duration_s * n) if n and duration_s else 0.0
    return clock, event, mean_rate, mon.count

sizes = [20, 40, 80]
rows = []
for n in sizes:
    clock, event, rate, spikes = run_toggles(n)
    sav = (1.0 - event / max(clock, 1)) * 100
    rows.append((n, clock, event, sav, rate, spikes))
    print(f"N={n:3d} clock={clock:8d} event={event:8d} savings={sav:5.1f}% rate≈{rate:.2f} Hz spikes={spikes}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ns = [r[0] for r in rows]
ax.plot(ns, [r[1] for r in rows], "s-", label="clock-driven toggles")
ax.plot(ns, [r[2] for r in rows], "o-", label="event-driven toggles")
ax.set_xlabel("N (Perfect Integrator neurons)")
ax.set_ylabel("toggle proxy count")
ax.set_title("Honest energy proxy — toggles, not joules")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print("NB-46 complete. No joule claim made.")
